<a href="https://colab.research.google.com/github/prat-kap/Assignment-Task-Management/blob/main/mlops_fundamentals_session_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MLOps Fundamentals — Session Notebook
### AIM Program | IIT Patna × Masai School

---

**Topic:** MLOps Fundamentals — Experiment Tracking, Model Versioning & Git Workflows  
**Estimated Time:** 90–110 minutes  
**Difficulty:** Intermediate  

---

## What You Will Build

By the end of this notebook, you will have built a **complete MLOps workflow from scratch**:

1. Instrument a training script with **MLflow** — log params, metrics, and artifacts
2. Run **multiple experiments** and compare them in the MLflow UI
3. Log a run with **Weights & Biases (W&B)** and explore the live dashboard
4. **Register** a trained model in the MLflow Model Registry
5. Transition a model through **lifecycle stages**: Staging → Production → Archived
6. Tag every run with its **Git commit hash** for full reproducibility
7. Compare **three model configurations** systematically and promote the best one

No GPU required — we use scikit-learn throughout.

---

## Learning Goals

- LO1: Explain what MLOps solves — reproducibility, collaboration, reliability
- LO2: Instrument a training script with MLflow to log params, metrics, and artifacts
- LO3: Use the MLflow Model Registry to version and stage trained models
- LO4: Apply W&B for real-time tracking and run comparison
- LO5: Structure a Git-linked workflow for reproducible ML experiments

---

## Section 0: Environment Setup

Install required libraries. This cell only needs to run once.

In [ ]:
# Install dependencies
!pip install -q mlflow scikit-learn matplotlib pandas wandb

In [ ]:
import os
import subprocess
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore")

print("Libraries loaded successfully.")
print(f"MLflow version: {mlflow.__version__}")

---
## Section 1: Dataset & Baseline

We use the **Iris classification dataset** — 150 samples across 3 species, 4 numeric features. It is small enough to train instantly, large enough to show meaningful metric differences across model configurations.

Our goal: build a **reproducible experiment pipeline** that logs every training run — not just the best one. This mirrors real ML projects where you compare dozens of configurations before choosing a model to deploy.

In [ ]:
# Load and inspect the dataset
X, y = load_iris(return_X_y=True)
feature_names = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
target_names  = ["setosa", "versicolor", "virginica"]

df = pd.DataFrame(X, columns=feature_names)
df["species"] = [target_names[i] for i in y]

print(f"Dataset shape: {X.shape}")
print(f"Features: {feature_names}")
print(f"Classes:  {target_names}")
print(f"Class distribution:\n{df['species'].value_counts()}")
df.head()

In [ ]:
# Create a reproducible train/validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set: {X_train.shape[0]} samples")
print(f"Val   set: {X_val.shape[0]} samples")
print(f"\nClass balance in train: {np.bincount(y_train)}")
print(f"Class balance in val:   {np.bincount(y_val)}")

**Key observation:** We use `stratify=y` to ensure each class is proportionally represented in both splits. Without this, a random split on 150 samples could produce imbalanced splits that skew metric comparisons across runs.

**Reproducibility note:** Fix `random_state=42` in every stochastic operation. When this notebook runs again next month — by you or a teammate — the split is identical and metrics are directly comparable.

---
## Section 2: MLflow Experiment Tracking — Core Concepts

Before writing any training code, we configure MLflow.

### The Tracking Hierarchy

```
Experiment  ("iris-classification")
└── Run  (one training job, one configuration)
    ├── Parameters   — what you set before training (hyperparameters)
    ├── Metrics      — what you measure after training (accuracy, F1)
    ├── Artifacts    — files produced (model file, plots, CSVs)
    └── Tags         — free-form labels (git commit, dataset version)
```

Every run gets a unique `run_id`. This ID is how you find, compare, and load any historical result.

In [ ]:
# Configure the MLflow tracking server
# By default, mlflow stores data locally in ./mlruns/
# For a team setup, point this at a remote server:
#   mlflow.set_tracking_uri("http://your-mlflow-server:5000")

mlflow.set_tracking_uri("sqlite:///mlflow.db")  # local SQLite backend

EXPERIMENT_NAME = "iris-classification"
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"Experiment '{EXPERIMENT_NAME}' ready.")
print("To explore runs in the UI: mlflow ui --backend-store-uri sqlite:///mlflow.db")

### 2.1 Your First Instrumented Training Run

In [ ]:
# ── Helper: capture git commit hash ──────────────────────────────────────────
def get_git_commit():
    """Return the current git commit hash, or 'no-git' if not in a repo."""
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL
        ).decode().strip()
    except (subprocess.CalledProcessError, FileNotFoundError):
        return "no-git"

# ── Helper: evaluate a trained model ─────────────────────────────────────────
def evaluate(model, X_val, y_val):
    """Return a dict of evaluation metrics."""
    preds = model.predict(X_val)
    return {
        "val_accuracy":  round(accuracy_score(y_val, preds),  4),
        "val_f1":        round(f1_score(y_val, preds, average="weighted"), 4),
        "val_precision": round(precision_score(y_val, preds, average="weighted"), 4),
        "val_recall":    round(recall_score(y_val, preds, average="weighted"), 4),
    }

print("Helper functions defined.")

In [ ]:
# ── First MLflow run: Random Forest baseline ─────────────────────────────────
params = {
    "n_estimators": 100,
    "max_depth":    5,
    "random_state": 42
}

with mlflow.start_run(run_name="rf-baseline") as run:

    # 1. Log parameters — always BEFORE training
    mlflow.log_params(params)

    # 2. Log tags — metadata linking this run to code and data
    mlflow.set_tag("git_commit",      get_git_commit())
    mlflow.set_tag("dataset_version", "iris-v1")
    mlflow.set_tag("model_family",    "random_forest")

    # 3. Train
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)

    # 4. Evaluate and log metrics — always AFTER training
    metrics = evaluate(model, X_val, y_val)
    mlflow.log_metrics(metrics)

    # 5. Log the model as an artifact
    mlflow.sklearn.log_model(model, artifact_path="random-forest-model")

    # 6. Log a confusion matrix plot
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_estimator(
        model, X_val, y_val, display_labels=target_names, ax=ax
    )
    plt.tight_layout()
    plt.savefig("confusion_matrix_rf_baseline.png", dpi=100)
    mlflow.log_artifact("confusion_matrix_rf_baseline.png")
    plt.close()

    run_id = run.info.run_id

print(f"Run ID:  {run_id}")
print(f"Metrics: {metrics}")
print("\nArtifacts logged: model + confusion_matrix.png")

> ⚠️ **Common Mistake — Parameter vs Metric timing:** Parameters describe *what you decided* before training. Metrics describe *what happened* during/after training. Always log params first, metrics last. Logging a metric before training (e.g., logging 0.0 accuracy) corrupts your experiment history.

> ⚠️ **Common Mistake — Using `log_artifact` for models:** Always use `mlflow.sklearn.log_model()` (not `log_artifact`) for trained models. The framework-specific logger stores the model signature, serialisation metadata, and a `MLmodel` manifest that the registry and serving layer require.

---
## Section 3: Running Multiple Experiments — Systematic Comparison

The real power of experiment tracking is comparing many runs. We now train three model families across multiple hyperparameter configurations and log every run. Then we use the MLflow API to find the best one programmatically.

In [ ]:
# Define all experiment configurations
EXPERIMENTS = [
    # Random Forest — vary tree count and depth
    {
        "run_name":    "rf-n50-d3",
        "model_class": RandomForestClassifier,
        "params":      {"n_estimators": 50,  "max_depth": 3, "random_state": 42},
        "family":      "random_forest"
    },
    {
        "run_name":    "rf-n100-d5",
        "model_class": RandomForestClassifier,
        "params":      {"n_estimators": 100, "max_depth": 5, "random_state": 42},
        "family":      "random_forest"
    },
    {
        "run_name":    "rf-n200-d8",
        "model_class": RandomForestClassifier,
        "params":      {"n_estimators": 200, "max_depth": 8, "random_state": 42},
        "family":      "random_forest"
    },
    # Gradient Boosting — vary learning rate
    {
        "run_name":    "gb-lr0.1",
        "model_class": GradientBoostingClassifier,
        "params":      {"learning_rate": 0.1, "n_estimators": 100, "random_state": 42},
        "family":      "gradient_boosting"
    },
    {
        "run_name":    "gb-lr0.05",
        "model_class": GradientBoostingClassifier,
        "params":      {"learning_rate": 0.05, "n_estimators": 150, "random_state": 42},
        "family":      "gradient_boosting"
    },
    # Logistic Regression — vary regularisation
    {
        "run_name":    "lr-C1.0",
        "model_class": LogisticRegression,
        "params":      {"C": 1.0,  "max_iter": 500, "random_state": 42},
        "family":      "logistic_regression"
    },
    {
        "run_name":    "lr-C0.1",
        "model_class": LogisticRegression,
        "params":      {"C": 0.1,  "max_iter": 500, "random_state": 42},
        "family":      "logistic_regression"
    },
]

print(f"Configured {len(EXPERIMENTS)} experiment runs.")

In [ ]:
# Run all experiments — each becomes one MLflow run
commit = get_git_commit()
run_ids = []

for exp in EXPERIMENTS:
    with mlflow.start_run(run_name=exp["run_name"]) as run:

        # Log parameters and tags
        mlflow.log_params(exp["params"])
        mlflow.set_tag("git_commit",      commit)
        mlflow.set_tag("dataset_version", "iris-v1")
        mlflow.set_tag("model_family",    exp["family"])

        # Train and evaluate
        model = exp["model_class"](**exp["params"])
        model.fit(X_train, y_train)
        metrics = evaluate(model, X_val, y_val)
        mlflow.log_metrics(metrics)

        # Log model
        mlflow.sklearn.log_model(model, artifact_path="model")

        run_ids.append(run.info.run_id)
        print(f"  {exp['run_name']:<20} val_accuracy={metrics['val_accuracy']}  val_f1={metrics['val_f1']}")

print(f"\n{len(run_ids)} runs logged to experiment '{EXPERIMENT_NAME}'.")
print("Open the MLflow UI to compare: mlflow ui --backend-store-uri sqlite:///mlflow.db")

**Observation:** Notice that all seven runs logged in seconds. Because every run is stored in the MLflow backend, you can:
- Sort by any metric to find the best configuration instantly
- Filter by `model_family` tag to compare within a family
- Plot any metric across all runs to visualise hyperparameter sensitivity

In the MLflow UI: select all runs → click **Compare** → see side-by-side tables and parallel coordinate plots.

In [ ]:
# Find the best run programmatically — no UI required
client = MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

# Search all runs in this experiment, order by val_accuracy descending
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.val_accuracy DESC"],
    max_results=10
)

print(f"{'Run Name':<25} {'val_accuracy':>13} {'val_f1':>10} {'Model Family':<20}")
print("-" * 75)
for r in runs:
    name   = r.data.tags.get("mlflow.runName", r.info.run_id[:8])
    acc    = r.data.metrics.get("val_accuracy", 0)
    f1     = r.data.metrics.get("val_f1", 0)
    family = r.data.tags.get("model_family", "unknown")
    print(f"{name:<25} {acc:>13.4f} {f1:>10.4f} {family:<20}")

best_run = runs[0]
print(f"\n✅ Best run: '{best_run.data.tags.get('mlflow.runName')}' — val_accuracy={best_run.data.metrics['val_accuracy']}")

---
## Section 4: MLflow Model Registry — Versioning and Lifecycle

The **Model Registry** adds lifecycle management on top of experiment tracking. Instead of referring to models by `run_id` (opaque), you refer to them by name and stage (`Production`, `Staging`).

This separation is critical in production: your serving infrastructure loads `models:/IrisClassifier/Production` — a stable pointer — while the underlying model version changes without any code modification.

In [ ]:
MODEL_NAME = "IrisClassifier"

# ── Register the best run's model ────────────────────────────────────────────
best_run_id  = best_run.info.run_id
model_uri    = f"runs:/{best_run_id}/model"

registered = mlflow.register_model(
    model_uri=model_uri,
    name=MODEL_NAME
)

print(f"Registered model:  '{MODEL_NAME}'")
print(f"Version:           {registered.version}")
print(f"Source run:        {best_run_id}")
print(f"Current stage:     {registered.current_stage}")

In [ ]:
# ── Inspect all registered versions ──────────────────────────────────────────
versions = client.get_latest_versions(MODEL_NAME, stages=["None", "Staging", "Production", "Archived"])

print(f"\nAll versions of '{MODEL_NAME}':")
print(f"{'Version':<10} {'Stage':<15} {'Run ID':<35}")
print("-" * 62)
for v in versions:
    print(f"{v.version:<10} {v.current_stage:<15} {v.run_id[:32]}")

In [ ]:
# ── Promote to Staging for QA testing ───────────────────────────────────────
client.transition_model_version_stage(
    name=MODEL_NAME,
    version=registered.version,
    stage="Staging",
    archive_existing_versions=False
)
print(f"Version {registered.version} → Staging")

# Simulate QA: load the staged model and run a final evaluation
staged_model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}/Staging")
staged_preds = staged_model.predict(X_val)
staged_acc   = accuracy_score(y_val, staged_preds)

print(f"QA evaluation on staged model: val_accuracy = {staged_acc:.4f}")

if staged_acc >= 0.90:
    print("✅ QA passed — promoting to Production")
else:
    print("❌ QA failed — model stays in Staging")

In [ ]:
# ── Promote to Production ────────────────────────────────────────────────────
client.transition_model_version_stage(
    name=MODEL_NAME,
    version=registered.version,
    stage="Production",
    archive_existing_versions=True   # automatically archives the previous Production version
)

print(f"Version {registered.version} → Production")
print(f"Previous Production version automatically → Archived")

# Verify
production_versions = client.get_latest_versions(MODEL_NAME, stages=["Production"])
print(f"\nCurrent Production version: {production_versions[0].version}")

In [ ]:
# ── Load the Production model — the way serving code does it ─────────────────
# Note: always load by stage name, NEVER by version number.
# This way, your serving code stays identical across model updates.

production_model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}/Production")
final_preds      = production_model.predict(X_val)
final_acc        = accuracy_score(y_val, final_preds)

print(f"Production model val_accuracy: {final_acc:.4f}")
print("\nThis is how your API server loads the model in production:")
print(f'  model = mlflow.sklearn.load_model("models:/{MODEL_NAME}/Production")')

> ⚠️ **Common Mistake — Hardcoding version numbers in serving code:**
> ```python
> # ❌ Wrong — requires a code change every time you promote a new model
> model = mlflow.sklearn.load_model("models:/IrisClassifier/3")
>
> # ✅ Correct — the registry stage pointer updates; your code never changes
> model = mlflow.sklearn.load_model("models:/IrisClassifier/Production")
> ```

> 💡 **The lifecycle in plain terms:** A new model starts in `None`. It goes to `Staging` when it's a candidate for production — QA tests run against it there. It goes to `Production` when QA passes. Old Production versions move to `Archived`, where they remain accessible for rollback but are no longer the "active" model.

---
## Section 5: Weights & Biases (W&B) — Cloud-Native Tracking

W&B provides experiment tracking as a managed cloud service. The core concepts — params, metrics, artifacts — are identical to MLflow. The differences are in infrastructure and UX.

**When to use W&B vs MLflow:**

| | MLflow | W&B |
|---|---|---|
| Hosting | Self-hosted (local or server) | Cloud (managed) |
| Real-time updates | No — requires UI refresh | Yes — live metric streaming |
| Team dashboards | Basic | Rich, interactive |
| Setup | `pip install` only | Requires `wandb login` |
| Cost | Free (open-source) | Free tier + paid plans |

In [ ]:
# ── W&B setup ────────────────────────────────────────────────────────────────
# Authenticate: wandb.login() will prompt for your API key if not set.
# Get a free key at https://wandb.ai
#
# For this notebook, we use offline mode so no account is needed.
# In a real project, remove the WANDB_MODE line and set your API key.

import wandb
os.environ["WANDB_MODE"] = "offline"   # remove this line when using a real account

wandb.login(anonymous="allow")
print("W&B ready. Mode:", os.environ.get("WANDB_MODE", "online"))

In [ ]:
# ── Instrument a training run with W&B ───────────────────────────────────────
run = wandb.init(
    project="iris-classification",         # equivalent to mlflow experiment name
    name="rf-n100-d5-wandb",               # equivalent to mlflow run name
    config={                               # equivalent to mlflow.log_params()
        "n_estimators": 100,
        "max_depth":    5,
        "random_state": 42,
        "dataset":      "iris-v1"
    }
)

# Train
model_wb = RandomForestClassifier(
    n_estimators=wandb.config.n_estimators,
    max_depth=wandb.config.max_depth,
    random_state=wandb.config.random_state
)
model_wb.fit(X_train, y_train)
metrics_wb = evaluate(model_wb, X_val, y_val)

# Log metrics — equivalent to mlflow.log_metrics()
wandb.log(metrics_wb)

# Log feature importances as a W&B table
importance_table = wandb.Table(
    columns=["feature", "importance"],
    data=[[f, round(float(i), 4)] for f, i in zip(feature_names, model_wb.feature_importances_)]
)
wandb.log({"feature_importance": importance_table})

# Log confusion matrix as a W&B plot
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_estimator(
    model_wb, X_val, y_val, display_labels=target_names, ax=ax
)
plt.tight_layout()
plt.savefig("cm_wandb.png", dpi=100)
wandb.log({"confusion_matrix": wandb.Image("cm_wandb.png")})
plt.close()

wandb.finish()

print(f"W&B run complete.")
print(f"Metrics: {metrics_wb}")
print("In online mode, open your W&B dashboard to see the run.")

**W&B-specific features to explore in the live dashboard (online mode):**

- **System panel:** GPU/CPU utilisation, memory usage — automatically captured, no code needed
- **Media panel:** your confusion matrix image rendered inline
- **Table view:** the feature importance table is interactive — sortable and filterable
- **Run comparison:** click any two runs → W&B builds a side-by-side diff of every config and metric
- **Parallel coordinates plot:** visualise how each hyperparameter correlates with each metric across all runs

---
## Section 6: Git Workflows for Reproducible ML

Experiment tracking records *what ran*. Git records *what code ran it*. Together they form a complete reproducibility chain: any result can be traced to an exact code state and reproduced from it.

### The Reproducibility Chain

```
MLflow run_id  →  git_commit tag  →  git checkout <hash>  →  exact code recovered
                ↓
           params + metrics  →  reproduce the result exactly
```

In [ ]:
# ── Demonstrate the full reproducibility chain ───────────────────────────────

def run_with_full_provenance(run_name, params, model_class, dataset_version="iris-v1"):
    """
    Train a model with complete provenance tracking:
    - MLflow records params, metrics, artifacts
    - Git commit hash links the run to exact code
    - Dataset version links the run to exact data
    Returns the run_id for registry use.
    """
    commit = get_git_commit()

    with mlflow.start_run(run_name=run_name) as run:
        # Provenance tags
        mlflow.set_tag("git_commit",      commit)
        mlflow.set_tag("dataset_version", dataset_version)
        mlflow.set_tag("model_family",    model_class.__name__)

        # Parameters
        mlflow.log_params(params)

        # Train
        model = model_class(**params)
        model.fit(X_train, y_train)

        # Metrics
        metrics = evaluate(model, X_val, y_val)
        mlflow.log_metrics(metrics)

        # 5-fold cross-validation score (logged as a metric)
        cv_scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
        mlflow.log_metric("cv_mean_accuracy", round(float(cv_scores.mean()), 4))
        mlflow.log_metric("cv_std_accuracy",  round(float(cv_scores.std()),  4))

        # Model + feature importance plot
        mlflow.sklearn.log_model(model, "model")

        if hasattr(model, "feature_importances_"):
            fig, ax = plt.subplots(figsize=(6, 3))
            ax.barh(feature_names, model.feature_importances_, color="steelblue")
            ax.set_xlabel("Importance")
            ax.set_title(f"Feature Importance — {run_name}")
            plt.tight_layout()
            plt.savefig(f"feat_imp_{run_name}.png", dpi=100)
            mlflow.log_artifact(f"feat_imp_{run_name}.png")
            plt.close()

        return run.info.run_id, metrics


# Run three fully-provenanced experiments
experiments_v2 = [
    ("rf-provenance-n100",  {"n_estimators": 100, "max_depth": 5, "random_state": 42}, RandomForestClassifier),
    ("gb-provenance-lr0.1", {"learning_rate": 0.1, "n_estimators": 100, "random_state": 42}, GradientBoostingClassifier),
    ("lr-provenance-C1",    {"C": 1.0, "max_iter": 500, "random_state": 42}, LogisticRegression),
]

provenance_runs = []
for run_name, params, cls in experiments_v2:
    rid, metrics = run_with_full_provenance(run_name, params, cls)
    provenance_runs.append((run_name, rid, metrics))
    print(f"  {run_name:<30} val_accuracy={metrics['val_accuracy']}  cv_mean={metrics.get('cv_mean_accuracy', 'N/A')}")

print(f"\n{len(provenance_runs)} fully-provenanced runs logged.")

In [ ]:
# ── Show the reproducibility information for each run ────────────────────────
print("\nReproducibility audit for each run:")
print("="*80)

for run_name, rid, _ in provenance_runs:
    r = client.get_run(rid)
    commit = r.data.tags.get("git_commit", "not recorded")
    dataset = r.data.tags.get("dataset_version", "not recorded")
    acc = r.data.metrics.get("val_accuracy", 0)

    print(f"\nRun:     {run_name}")
    print(f"  run_id:          {rid[:16]}...")
    print(f"  git_commit:      {commit[:16]}...")
    print(f"  dataset_version: {dataset}")
    print(f"  val_accuracy:    {acc}")
    print(f"  → To reproduce:  git checkout {commit[:8]} && python train.py")

### Recommended `.gitignore` for ML Projects

Add this to your repository root before your first commit:

```gitignore
# MLflow tracking data
mlruns/
mlartifacts/
mlflow.db

# Trained model files — track via registry, not Git
*.pkl
*.pt
*.h5
*.onnx
*.joblib

# Data — track with DVC or S3, not Git
data/raw/
data/processed/

# Environment and credentials
.env
secrets/
*.key

# Python
__pycache__/
*.pyc
.venv/
env/
```

### Recommended Branch Convention

```bash
main                          # stable, production-ready code
experiment/rf-baseline        # isolated experiment branch
experiment/gb-lr-sweep        # hyperparameter sweep  
experiment/feature-v2-trial   # new feature engineering approach

# After a successful experiment:
git checkout main
git merge experiment/rf-baseline
git tag -a "model-v1.2" -m "94% val acc — IrisClassifier v2 in Production"
git push origin --tags
```

---
## Section 7: Full MLOps Pipeline — Putting It All Together

We now assemble a complete, end-to-end `MLOpsPipeline` class that:
1. Runs a configurable set of experiments
2. Tracks every run with full provenance
3. Identifies the best run by a specified metric
4. Registers and promotes the best model to Production automatically

In [ ]:
class MLOpsPipeline:
    """
    A minimal, end-to-end MLOps pipeline.

    Phases:
      run_experiments()   — train all configs, log every run
      get_best_run()      — find best run by metric
      promote_best()      — register and stage the best model
    """

    def __init__(self, experiment_name, model_name):
        self.experiment_name = experiment_name
        self.model_name      = model_name
        self.client          = MlflowClient()
        mlflow.set_experiment(experiment_name)

    # ── Phase 1: Run all experiment configurations ────────────────
    def run_experiments(self, configs, X_train, y_train, X_val, y_val):
        """Train each config and log a fully-provenanced MLflow run."""
        commit     = get_git_commit()
        self.runs_ = []

        for cfg in configs:
            with mlflow.start_run(run_name=cfg["name"]) as run:
                mlflow.log_params(cfg["params"])
                mlflow.set_tag("git_commit",      commit)
                mlflow.set_tag("dataset_version", cfg.get("dataset", "iris-v1"))
                mlflow.set_tag("model_family",    cfg["model"].__name__)

                model = cfg["model"](**cfg["params"])
                model.fit(X_train, y_train)
                metrics = evaluate(model, X_val, y_val)
                mlflow.log_metrics(metrics)
                mlflow.sklearn.log_model(model, "model")

                self.runs_.append({
                    "name":    cfg["name"],
                    "run_id":  run.info.run_id,
                    "metrics": metrics
                })
        return self

    # ── Phase 2: Identify best run ────────────────────────────────
    def get_best_run(self, metric="val_accuracy"):
        """Return the run with the highest value of `metric`."""
        experiment = self.client.get_experiment_by_name(self.experiment_name)
        runs = self.client.search_runs(
            experiment_ids=[experiment.experiment_id],
            order_by=[f"metrics.{metric} DESC"],
            max_results=1
        )
        self.best_run_ = runs[0]
        return self.best_run_

    # ── Phase 3: Register and promote ────────────────────────────
    def promote_best(self, qa_threshold=0.90):
        """Register the best model and promote it to Production if it passes QA."""
        run_id    = self.best_run_.info.run_id
        model_uri = f"runs:/{run_id}/model"

        version = mlflow.register_model(model_uri, self.model_name)

        # QA check on the staged model
        self.client.transition_model_version_stage(
            name=self.model_name, version=version.version, stage="Staging"
        )
        staged_model = mlflow.sklearn.load_model(f"models:/{self.model_name}/Staging")
        qa_acc       = accuracy_score(y_val, staged_model.predict(X_val))

        if qa_acc >= qa_threshold:
            self.client.transition_model_version_stage(
                name=self.model_name, version=version.version,
                stage="Production", archive_existing_versions=True
            )
            status = f"✅ Promoted to Production (QA acc: {qa_acc:.4f})"
        else:
            status = f"❌ Stayed in Staging (QA acc: {qa_acc:.4f} < threshold {qa_threshold})"

        return version.version, status


# ── Run the full pipeline ─────────────────────────────────────────────────────
pipeline_configs = [
    {"name": "pipeline-rf-n100",  "model": RandomForestClassifier,    "params": {"n_estimators": 100, "max_depth": 5,  "random_state": 42}},
    {"name": "pipeline-rf-n200",  "model": RandomForestClassifier,    "params": {"n_estimators": 200, "max_depth": 8,  "random_state": 42}},
    {"name": "pipeline-gb-lr0.1", "model": GradientBoostingClassifier,"params": {"learning_rate": 0.1, "n_estimators": 100, "random_state": 42}},
    {"name": "pipeline-lr-C1.0",  "model": LogisticRegression,         "params": {"C": 1.0, "max_iter": 500, "random_state": 42}},
]

pipeline = MLOpsPipeline(
    experiment_name="iris-pipeline",
    model_name="IrisClassifier-Pipeline"
)

pipeline.run_experiments(pipeline_configs, X_train, y_train, X_val, y_val)
best = pipeline.get_best_run(metric="val_accuracy")
version, status = pipeline.promote_best(qa_threshold=0.90)

print(f"Best run:  {best.data.tags.get('mlflow.runName')}")
print(f"Best acc:  {best.data.metrics['val_accuracy']}")
print(f"Registry:  IrisClassifier-Pipeline v{version}")
print(f"Status:    {status}")

---
## Section 8: Summary and Exercises

### What We Built

| Component | Implementation |
|---|---|
| Experiment setup | `mlflow.set_experiment()` — named experiment groups |
| Parameter logging | `mlflow.log_params()` — before training |
| Metric logging | `mlflow.log_metrics()` — after training |
| Artifact logging | `mlflow.sklearn.log_model()` + `log_artifact()` |
| Provenance tags | `git_commit`, `dataset_version`, `model_family` |
| Run comparison | `client.search_runs()` with `order_by` |
| W&B tracking | `wandb.init()`, `wandb.config`, `wandb.log()` |
| Model Registry | `mlflow.register_model()` — named versioned models |
| Lifecycle stages | `None → Staging → Production → Archived` |
| Production loading | `mlflow.sklearn.load_model("models:/Name/Production")` |
| Full pipeline | `MLOpsPipeline` — experiments → best run → registry |

### Key Insights

1. **Tracking is not overhead — it is the experiment.** A training run without logging is like a science experiment without a lab notebook: unrepeatable and unverifiable.

2. **Parameters and metrics have different timing.** Log parameters before training (they are your inputs). Log metrics after training (they are your outputs). Mixing this up corrupts your experiment history.

3. **The Git commit tag closes the reproducibility loop.** MLflow gives you params and metrics. The git commit gives you the exact code. Together any result can be reproduced perfectly — months or years later.

4. **Load production models by stage, never by version.** `models:/IrisClassifier/Production` is a stable pointer. `models:/IrisClassifier/3` is a specific version that becomes stale the moment you deploy v4.

5. **W&B and MLflow solve the same problem differently.** MLflow is self-hosted and infrastructure-agnostic. W&B is cloud-native with richer real-time collaboration. Choose based on your team's needs, not popularity.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# EXERCISES — Complete these to test your understanding
# ─────────────────────────────────────────────────────────────────

# Exercise 1: Add a new model family to the pipeline.
# Add an ExtraTreesClassifier configuration to `pipeline_configs`
# with n_estimators=150, max_depth=6, random_state=42.
# Re-run the pipeline and check if it becomes the best model.

# YOUR CODE HERE


# Exercise 2: Log metrics at each cross-validation fold, not just the mean.
# Modify `run_with_full_provenance()` to log val_accuracy_fold_1 through
# val_accuracy_fold_5 as separate metrics using a loop.
# Hint: use cross_val_score(..., cv=5) and enumerate the scores.

# YOUR CODE HERE


# Exercise 3: Add a model comparison report as an artifact.
# After running all experiments, create a pandas DataFrame summarising
# run_name, val_accuracy, val_f1, and git_commit for all runs.
# Save it as 'experiment_summary.csv' and log it as an artifact
# on a new MLflow run called "experiment-summary".

# YOUR CODE HERE


# Exercise 4 (Challenge): Implement automatic rollback.
# After promoting the best model to Production, simulate a production
# failure by asserting val_accuracy < 0.50 (force-fail).
# Implement a rollback() method on MLOpsPipeline that:
#   1. Identifies the most recent Archived version
#   2. Transitions it back to Production
#   3. Transitions the current Production version to Archived
# Print the version numbers before and after rollback.

# YOUR CODE HERE


print("Exercises ready — begin coding!")